In [1]:
import numpy as np
import pandas as pd

# ── Session Data ──────────────────────────────────────────────────────────────
easee_sessions = pd.read_parquet(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\filtered_easee_sessions.parquet')
site_data = pd.read_parquet(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\filtered_easee_sites.parquet')
easee_sessions['carConnected']    = pd.to_datetime(easee_sessions['carConnected'], utc=True)
easee_sessions['carDisconnected'] = pd.to_datetime(easee_sessions['carDisconnected'], utc=True)

# ── Sessions per site ID with min/max carConnected ────────────────────────────
sessions_per_site = (
    easee_sessions
    .groupby('site_id')
    .agg(
        session_count = ('site_id',      'size'),
        first_session = ('carConnected', 'min'),
        last_session  = ('carConnected', 'max'),
    )
    .reset_index()
    .sort_values('session_count', ascending=False)
)

sessions_per_site = sessions_per_site.merge(
    site_data[['id']].drop_duplicates(),
    left_on='site_id', right_on='id',
    how='left'
).drop(columns='id')

print(sessions_per_site[['site_id', 'session_count', 'first_session', 'last_session']].to_string(index=False))

pdf = easee_sessions[easee_sessions['site_id'] == 419588]

# ── Price Data ──────────────────────────────────────────────────────────────
df_dyn_tarif = pd.read_csv(r'C:\Users\ckw-ThCa\OneDrive - CKW-Gruppe\Origination-General\Team\Caspar\Model_HEMS\hems_resopt\data\01_raw\home_dynamic_Leistungspreis_CKW_25.csv')
df_dyn_tarif['Timestamp'] = pd.to_datetime(df_dyn_tarif['Timestamp'], dayfirst=True, errors='coerce')
df_dyn_tarif['Timestamp'] = df_dyn_tarif['Timestamp'].dt.tz_localize('Europe/Zurich', ambiguous='infer')
df_dyn_tarif['Timestamp'] = df_dyn_tarif['Timestamp'].dt.tz_convert('UTC')
df_dyn_tarif = df_dyn_tarif.set_index('Timestamp', drop=True).sort_index()

# ── Time Index ─────────────────────────────────────────────────────────────────
start_time = pd.Timestamp('2025-01-01', tz='UTC')
end_time   = pd.Timestamp('2026-01-01', tz='UTC')
idx        = pd.date_range(start=start_time, freq='15min', end=end_time, tz='UTC')

# ── Optimization Dataframe -────────────────────────────────────────────────────
columns      = ['power_min', 'power_max', 'e_in', 'e_out', 'e_cap']
df_ev_inputs = pd.DataFrame(index=idx, columns=columns, dtype=float)
df_ev_inputs.index = pd.to_datetime(df_ev_inputs.index, format='mixed', dayfirst=True)

# ── Assign to df_ev_inputs ─────────────────────────────────────────────────────
df_ev_inputs['dyn_tarif'] = df_dyn_tarif['home dynamic']
df_ev_inputs['dyn_tarif'] = df_ev_inputs['dyn_tarif'] / 100 # From Rp to CHF 
df_ev_inputs['dyn_tarif'] = df_ev_inputs['dyn_tarif'].ffill()

 site_id  session_count             first_session              last_session
  300051           2393 2024-12-31 16:34:08+00:00 2026-04-29 12:56:43+00:00
  275947           1603 2024-12-30 15:13:26+00:00 2026-04-29 20:00:11+00:00
  287032            920 2024-12-31 18:11:07+00:00 2026-04-28 16:13:59+00:00
  405891            900 2024-12-31 22:07:17+00:00 2026-04-29 17:21:52+00:00
  408561            371 2025-01-03 08:12:01+00:00 2026-04-25 21:59:36+00:00
  419588            329 2024-12-31 12:50:05+00:00 2026-04-28 19:18:06+00:00
  328456            251 2025-01-02 09:38:44+00:00 2026-04-29 15:53:58+00:00
  587263            212 2025-01-05 13:40:54+00:00 2026-04-25 12:41:51+00:00
  263264            192 2025-01-01 10:30:08+00:00 2026-04-27 15:33:03+00:00
  226605            184 2025-01-01 14:58:42+00:00 2026-04-27 17:00:12+00:00
  585442            126 2025-01-06 19:22:12+00:00 2026-04-24 15:34:37+00:00
  667659             53 2025-10-14 16:43:56+00:00 2026-04-29 16:56:20+00:00


In [2]:
def build_expensive_hour_mask(
    df: pd.DataFrame,
    price_col: str,
    n_hours: int,
) -> pd.Series:
    """Return a single 0/1 mask Series blocking the n most expensive hours/day."""
    if not isinstance(df.index, pd.DatetimeIndex):
        raise KeyError("DataFrame must have a DatetimeIndex.")

    day = df.index.date
    hour = df.index.hour

    hourly_price = (
        df.groupby([day, hour])[price_col]
        .mean()
        .rename_axis(['day', 'hour'])
        .reset_index()
    )

    expensive_hours = (
        hourly_price
        .groupby('day')
        .apply(lambda g: set(g.nlargest(n_hours, price_col)['hour']))
    )

    mask = pd.Series(
        [0 if h in expensive_hours.loc[d] else 1 for d, h in zip(day, hour)],
        index=df.index,
        name=f'{price_col}_mask_{n_hours}',
    )
    return mask


In [3]:
from hems_resopt.utils.pre_process_ev import (prepare_sessions, build_connection_df, 
                        get_power_bounds, get_e_in_out_capacity,
                        build_input_dict)

battery_sizes = 80
power_max_per_charger = 11
EV_opt_name='EV_Fleet'
columns      = ['power_min', 'power_max', 'e_in', 'e_out', 'energy_capacity']
per_charger_mode = True

# ── Assemble df_ev_inputs with one set of columns per charger ─────────────────

# Step 1 — prepare sessions
sessions = prepare_sessions(pdf, start_time, end_time)
sessions = sessions.drop_duplicates()

charger_ids = sessions['chargerId'].unique().tolist()

battery_dict = build_input_dict(charger_ids, battery_sizes)
power_dict   = build_input_dict(charger_ids, power_max_per_charger)

# Step 2 — build the central connection matrix
connection_df = build_connection_df(sessions, idx, charger_ids)

# Step 3 — per-charger power bounds
df_power_bounds = get_power_bounds(connection_df, power_dict, per_charger=per_charger_mode)

# Step 4 — per-charger e_in / e_out / e_cap
df_energy_bounds, capped_kWh_list = get_e_in_out_capacity(
    sessions, connection_df, idx, battery_dict, power_dict, per_charger=per_charger_mode
)

# Step 5 — merge into a single wide df_ev_inputs
df = pd.concat([df_power_bounds, df_energy_bounds], axis=1)
df_ev_inputs = pd.concat([df_ev_inputs, df], axis=1)  

# Step 6 — apply expensive-hour mask PER CHARGER
mask_blocked_hours = build_expensive_hour_mask(
    df_ev_inputs,
    price_col='dyn_tarif',
    n_hours=8,
)

for cid in charger_ids:
    df_ev_inputs[f'power_min_{cid}'] = df_ev_inputs[f'power_min_{cid}'] * mask_blocked_hours


INFO: Total kWh removed by power feasibility cap: 35.66 kWh across 6 sessions.


In [4]:
import pyomo.environ as pyo
from hems_resopt.components.grid import GridPeakShave
from res_opt_core import EnergyModel, Grid, AuctionMarket, plot_battery_operation
from hems_resopt.components.ev import EV
from pyomo.contrib.solver.solvers.highs import Highs


# ── Solver setup ────────────────────────────────────────────────────────────
custom_solver = Highs()
custom_solver.highs_options = {
    "primal_feasibility_tolerance": 1e-4,   # default: 1e-7  (loosen)
    "dual_feasibility_tolerance":   1e-4,   # default: 1e-7  (loosen)
    "ipm_optimality_tolerance":     1e-4,   # interior point tolerance
    "time_limit":                   300.0,  # seconds — increase if needed
    "presolve":                     "on",   # keep presolve active
    "solver":                       "simplex",  # or "ipm" for interior point
}

# ── Model ─────────────────────────────────────────────────────────────────────
model = EnergyModel(
    num_steps=len(df_ev_inputs.index),
    slot_length="15min",
    solver=custom_solver,
    timestamp=df_ev_inputs.index,
)

# ── Build one EV component per charger (modular) ───────────────────────────────
# If per_charger_mode is False, charger_ids should be [EV_opt_name] and the
# corresponding columns in df_ev_inputs should be the plain pool-level names
# ('power_min', 'power_max', 'e_cap', 'e_in', 'e_out') for this to still work
# seamlessly — see the column-name resolution below.


if per_charger_mode:
    ev_ids = charger_ids   # list of charger IDs, e.g. from sessions['chargerId'].unique()
else:
    ev_ids = [EV_opt_name]  # single pooled component

ev_components = []

for cid in ev_ids:
    suffix = f'_{cid}' if per_charger_mode else ''

    power_min_col = f'power_min{suffix}'
    power_max_col = f'power_max{suffix}'
    e_cap_col     = f'e_cap{suffix}'
    e_in_col      = f'e_in{suffix}'
    e_out_col     = f'e_out{suffix}'

    required_cols = [power_min_col, power_max_col, e_cap_col, e_in_col, e_out_col]
    missing = [c for c in required_cols if c not in df_ev_inputs.columns]
    if missing:
        raise KeyError(f"Missing columns in df_ev_inputs for '{cid}': {missing}")

    ev = EV(
        name=f'EV_{cid}' if per_charger_mode else EV_opt_name,
        power_nominal=power_dict.get(cid, 200) if per_charger_mode else 200,
        power_min=df_ev_inputs[power_min_col],
        power_max=df_ev_inputs[power_max_col],
        energy_capacity=df_ev_inputs[e_cap_col],
        soc_initial=0.0,
        soc_final=None,
        energy_in_slot_start=df_ev_inputs[e_in_col],
        energy_out_slot_end=df_ev_inputs[e_out_col],
        # Make the HARD soc_min/soc_max bounds temporarily soft to diagnose infeasibility
        cost_min_energy_violation=100,   # CHF/kWh - high, so only used if truly forced
        cost_max_energy_violation=100,
    )
    ev_components.append(ev)
    model.add_component(ev)

# ── Market ──────────────────────────────────────────────────────────────────
dynamischer_tariff = AuctionMarket(
    name='dynamic_tariff',
    price_curve=df_ev_inputs['dyn_tarif'],
    # market_time_unit='1hr' definiert die Auflösung des Marktes
)
model.add_component(dynamischer_tariff)

# ── Grid — now references ALL EV components, not just one ─────────────────────
grid = GridPeakShave(
    name='Grid_with_peakshave',
    assets=ev_components,          # modular list, works for 1..N chargers
    markets=[dynamischer_tariff],
    peak_power_price=1.5,          # CHF/kW
    time_horizon_peak='month_daylight_saving',
)
model.add_component(grid)

# ── Run ───────────────────────────────────────────────────────────────────────
model.build_and_run(silent=False)  # solver output visible

df_results = model.results.timeseries_to_pandas()


Running HiGHS 1.14.0 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
MIP has 1927255 rows; 1401640 cols; 3223972 nonzeros; 140164 integer variables (140164 binary)
Coefficient ranges:
  Matrix  [3e-03, 1e+01]
  Cost    [3e-04, 1e+02]
  Bound   [1e+00, 1e+00]
  RHS     [2e-02, 1e+01]
Presolving model
223457 rows, 129628 cols, 500781 nonzeros 1s
179940 rows, 105924 cols, 478802 nonzeros 2s
111327 rows, 75526 cols, 330256 nonzeros 3s
92176 rows, 63393 cols, 274241 nonzeros 4s
80123 rows, 51885 cols, 232230 nonzeros 4s
67950 rows, 40422 cols, 188340 nonzeros 5s
62576 rows, 36081 cols, 172958 nonzeros 5s
Presolve reductions: rows 62576(-1864679); columns 36081(-1365559); nonzeros 172958(-3051014) 

Solving MIP model with:
   62576 rows
   36081 cols (0 binary, 0 integer, 0 implied int., 36081 continuous, 0 domain fixed)
   172958 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P

In [5]:
# plot_battery_operation(df_results.index, ev_fleet, {"dynamischer_tariff": df_ev_inputs['dyn_tarif']})
# df_results

In [6]:
import re

# ── Discover all charger/EV names present in df_results ───────────────────────
ev_names = sorted({
    m.group(1)
    for c in df_results.columns
    if (m := re.match(r'^(.*)__var_power$', c))
})

# Exclude non-EV components (e.g. the market component)
ev_names = [name for name in ev_names if name != 'dynamic_tariff']

# ── Sum of min/max energy violations per charger ───────────────────────────────
eps = 1e-9

for ev_name in ev_names:
    min_col = f'{ev_name}__var_min_energy_violation'
    max_col = f'{ev_name}__var_max_energy_violation'

    if min_col not in df_results.columns or max_col not in df_results.columns:
        print(f"⚠️  {ev_name}: violation columns not found — skipping.")
        continue

    min_viol = df_results[min_col]
    max_viol = df_results[max_col]

    total_min = min_viol[min_viol.notna() & (min_viol.abs() > eps)].sum()
    total_max = max_viol[max_viol.notna() & (max_viol.abs() > eps)].sum()

    print(f"{ev_name}: min_violation = {total_min:.4f} kWh, max_violation = {total_max:.4f} kWh, total = {total_min + total_max:.4f} kWh")


EV_EC7LUYA2: min_violation = 221.9845 kWh, max_violation = 0.0000 kWh, total = 221.9845 kWh
EV_ECYFEKSY: min_violation = 0.0000 kWh, max_violation = 0.0000 kWh, total = 0.0000 kWh


In [7]:
from hems_resopt.utils.post_process_ev import plot_charger_usage, plot_representative_week, compute_ev_optimization_summary, print_summary
ladetarif_ckw = 0.33 # CHF/kWh total alles inklusive, energie, netz, lastpitzenkosten deckung, 
lastspitzekoste_ckw = 0.0587 # CHF/kWh (Annahme Verteilung Lastspitzenkosten: bei 6000kWh jährlich und monatlicher Spitzenleistung von 14.7 kW)
peak_power_price_dyn = 1
peak_power_price_stat = 1.5

df_post_process, summary_dict, monthly_df = compute_ev_optimization_summary(
    df_results=df_results,
    df_ev_inputs=df_ev_inputs,
    idx=idx,
    sessions=sessions,
    capped_kWh_list=capped_kWh_list,
    peak_power_price_dyn=peak_power_price_dyn,
    peak_power_price_stat=peak_power_price_stat,
    energy_costs=0.11,
    fix_costs= 0.07226,
    EV_opt_name=EV_opt_name
)
print_summary(summary_dict)
plot_charger_usage(easee_sessions, df_post_process, idx, connection_df, sessions, session_kWh_col='kiloWattHours')
plot_representative_week(df_ev_inputs['dyn_tarif'], df_post_process['energy_charged'])

UnboundLocalError: cannot access local variable 'tot_min_energy_viol' where it is not associated with a value

In [ ]:
# Plot the SoC overtime 
# see how much is charged / when violation is happening
# count the violations and the amount of energy per session which is not charged (e_cap at timestep in relation to SoC of timestep)

# Plot the load per timestep to see peak load of the month / week etc. 
# EV_Fleet__var_power -> negative is charging = load

In [ ]:
# 